In [7]:
import pandas as pd
import numpy as np
import re
import xarray as xr
import warnings

warnings.filterwarnings('ignore')

In [8]:
df = pd.read_excel('Dane/PositionReport.xlsx', header=[8,9])

In [9]:
czyste_kolumny = []

for col in df.columns:
    czesc_gorna = str(col[0]) if 'Unnamed' not in str(col[0]) else ''
    czesc_dolna = str(col[1]) if 'Unnamed' not in str(col[1]) else ''
    pelna_nazwa = f"{czesc_gorna} {czesc_dolna}".replace('\n', ' ').strip()
    pelna_nazwa = pelna_nazwa.replace('  ', ' ')
    czyste_kolumny.append(pelna_nazwa)
df.columns = czyste_kolumny



In [10]:

def dms_to_dd(dms_str):
    if pd.isna(dms_str):
        return np.nan
    match = re.match(r"(\d+)°\s+([\d.]+)'\s+([NESW])", str(dms_str))
    if not match:
        return np.nan 
    degrees = float(match.group(1))
    minutes = float(match.group(2))
    direction = match.group(3)
    dd = degrees + (minutes / 60.0)
    if direction in ['S', 'W']:
        dd *= -1
        
    return round(dd, 5)
df.columns = df.columns.str.strip()
df['Time'] = pd.to_datetime(df['Time'], format='%d.%m.%Y %H:%M')
df['Lat_dd'] = df['Lat'].apply(dms_to_dd)
df['Lon_dd'] = df['Lon'].apply(dms_to_dd)

df_clean = df.dropna(subset=['Speed [kn]']).copy()
df_clean = df_clean[df_clean['Speed [kn]'] > 0]
print(f"Liczba wierszy przed czyszczeniem: {len(df)}")
print(f"Liczba wierszy po odcięciu postojów: {len(df_clean)}")
display(df_clean[['Time', 'Lat_dd', 'Lon_dd', 'Speed [kn]', 'Course [°]']].head())

Liczba wierszy przed czyszczeniem: 156942
Liczba wierszy po odcięciu postojów: 132042


,Time,Lat_dd,Lon_dd,Speed [kn],Course [°]
0,2023-02-02 00:06:00,56.82667,-0.12833,15.0,357.0
1,2023-02-02 00:21:00,56.88667,-0.12500,15.0,47.0
2,2023-02-02 00:33:00,56.92333,-0.05833,16.0,46.0
3,2023-02-02 00:54:00,56.98667,0.05500,15.0,44.0
4,2023-02-02 01:15:00,57.05000,0.17500,16.0,46.0


In [11]:
print("Bounding Box:")
print(f"Północ (North): {df_clean['Lat_dd'].max()}")
print(f"Południe (South): {df_clean['Lat_dd'].min()}")
print(f"Wschód (East): {df_clean['Lon_dd'].max()}")
print(f"Zachód (West): {df_clean['Lon_dd'].min()}")
print(f"Początek (Od): {df_clean['Time'].min()}")
print(f"Koniec (Do): {df_clean['Time'].max()}")

Bounding Box:
Północ (North): 61.23332
Południe (South): 51.03923
Wschód (East): 11.20088
Zachód (West): -2.12149
Początek (Od): 2023-02-02 00:06:00
Koniec (Do): 2026-02-02 08:34:57


In [12]:
df_clean.head()

,Time,Lat,Lon,Speed [kn],Calculated Speed [kn],Course [°],Distance since last point [nm],Total distance run [nm],Lat_dd,Lon_dd
0,2023-02-02 00:06:00,56° 49.6000' N,000° 07.7000' W,15.0,NaN,357.0,0.000000,0.000000,56.82667,-0.12833
1,2023-02-02 00:21:00,56° 53.2000' N,000° 07.5000' W,15.0,14.4,47.0,3.608126,3.608126,56.88667,-0.12500
2,2023-02-02 00:33:00,56° 55.4000' N,000° 03.5000' W,16.0,15.5,46.0,3.105603,6.713728,56.92333,-0.05833
3,2023-02-02 00:54:00,56° 59.2000' N,000° 03.3000' E,15.0,15.2,44.0,5.318924,12.032653,56.98667,0.05500
4,2023-02-02 01:15:00,57° 03.0000' N,000° 10.5000' E,16.0,15.6,46.0,5.468927,17.501580,57.05000,0.17500


In [13]:

df_clean['Time'] = pd.to_datetime(df_clean['Time'], utc=True).dt.tz_localize(None)

sciezka_do_plikow = 'Dane/*.nc'
ds_era5 = xr.open_mfdataset(sciezka_do_plikow, combine='by_coords', engine='netcdf4')

if ds_era5.longitude.max() > 180:
    ds_era5.coords['longitude'] = (ds_era5.coords['longitude'] + 180) % 360 - 180
    ds_era5 = ds_era5.sortby(ds_era5.longitude)

t = xr.DataArray(df_clean['Time'], dims="z")
lat = xr.DataArray(df_clean['Lat_dd'], dims="z")
lon = xr.DataArray(df_clean['Lon_dd'], dims="z")

ds_wind = ds_era5[['u10', 'v10']].interp(valid_time=t, latitude=lat, longitude=lon, method='linear')
ds_wind = ds_wind.compute()

ds_waves_grid = ds_era5[['swh', 'mwd']]
ds_waves_grid = ds_waves_grid.ffill(dim='longitude', limit=2).bfill(dim='longitude', limit=2)
ds_waves_grid = ds_waves_grid.ffill(dim='latitude', limit=2).bfill(dim='latitude', limit=2)

ds_waves = ds_waves_grid.interp(valid_time=t, latitude=lat, longitude=lon, method='nearest')
ds_waves = ds_waves.compute()

df_clean['Wind_U_10m'] = ds_wind['u10'].values
df_clean['Wind_V_10m'] = ds_wind['v10'].values
df_clean['Wind_Speed_m_s'] = np.sqrt(df_clean['Wind_U_10m']**2 + df_clean['Wind_V_10m']**2)

df_clean['Wave_Height_Sig'] = ds_waves['swh'].values
df_clean['Wave_Dir_Mean'] = ds_waves['mwd'].values

df_clean['Wave_Height_Sig'] = df_clean['Wave_Height_Sig'].interpolate(method='linear', limit=12).fillna(0)

dir_rad = np.deg2rad(df_clean['Wave_Dir_Mean'])
df_clean['dir_sin'] = np.sin(dir_rad)
df_clean['dir_cos'] = np.cos(dir_rad)

df_clean['dir_sin'] = df_clean['dir_sin'].interpolate(method='linear', limit=12)
df_clean['dir_cos'] = df_clean['dir_cos'].interpolate(method='linear', limit=12)

df_clean['dir_sin'] = df_clean['dir_sin'].ffill().bfill()
df_clean['dir_cos'] = df_clean['dir_cos'].ffill().bfill()

interpolated_dir_rad = np.arctan2(df_clean['dir_sin'], df_clean['dir_cos'])
df_clean['Wave_Dir_Mean'] = (np.rad2deg(interpolated_dir_rad) + 360) % 360

df_clean['Wave_Dir_Sin'] = df_clean['dir_sin']
df_clean['Wave_Dir_Cos'] = df_clean['dir_cos']

df_clean = df_clean.drop(columns=['dir_sin', 'dir_cos'])

kolumny_wiatr = ['Wind_U_10m', 'Wind_V_10m', 'Wind_Speed_m_s']
df_clean[kolumny_wiatr] = df_clean[kolumny_wiatr].ffill().bfill()

print("\n--- Połączone i Oczyszczone Dane ---")
display(df_clean[['Time', 'Wave_Height_Sig', 'Wave_Dir_Mean', 'Wave_Dir_Sin', 'Wave_Dir_Cos','Wind_U_10m','Wind_V_10m']].head())

print("\nPodsumowanie zbioru (Liczba braków NaN):")
stats = df_clean[['Wave_Height_Sig', 'Wave_Dir_Mean', 'Wind_Speed_m_s', 'Wave_Dir_Sin', 'Wave_Dir_Cos']].isna().sum()
print(stats)


OSError: [Errno -51] NetCDF: Unknown file format: 'c:\\Users\\Mateusz\\Desktop\\rotooory\\projekt-rotory\\Dane\\mean_wave_direction_1.nc'

In [ ]:
def dms_to_dd(dms_str):
    if pd.isna(dms_str):
        return np.nan
    match = re.match(r"(\d+)°\s+([\d.]+)'\s+([NESW])", str(dms_str))
    if not match:
        return np.nan 
    degrees = float(match.group(1))
    minutes = float(match.group(2))
    direction = match.group(3)
    dd = degrees + (minutes / 60.0)
    if direction in ['S', 'W']:
        dd *= -1
    return round(dd, 5)

def calculate_awa_improved(df):
    df_calc = df.copy()
    
    V_s = df_calc['Speed [kn]'] * 0.514444
    
    course_rad = np.radians(df_calc['Course [°]'])
    df_calc['u_s'] = V_s * np.sin(course_rad)
    df_calc['v_s'] = V_s * np.cos(course_rad)
    
    df_calc['u_a'] = df_calc['Wind_U_10m'] - df_calc['u_s']
    df_calc['v_a'] = df_calc['Wind_V_10m'] - df_calc['v_s']    
    
    df_calc['AWA_north'] = np.degrees(np.arctan2(df_calc['u_a'], df_calc['v_a']))   
    df_calc['AWA_relative'] = (df_calc['AWA_north'] - df_calc['Course [°]'] + 180) % 360 - 180   
    
    df_calc['Apparent_Wind_Speed_m_s'] = np.sqrt(df_calc['u_a']**2 + df_calc['v_a']**2)
    
    return df_calc
    

In [ ]:
import glob
import os

currents_path = 'Dane/currents/*.nc'
ds_currents = xr.open_mfdataset(currents_path, combine='by_coords', engine='netcdf4')

sea_level_path = 'Dane/sea_level/*.nc'
ds_ssh_raw = xr.open_mfdataset(sea_level_path, combine='by_coords', engine='netcdf4')

if ds_currents.longitude.max() > 180:
    ds_currents.coords['longitude'] = (ds_currents.coords['longitude'] + 180) % 360 - 180
    ds_currents = ds_currents.sortby(ds_currents.longitude)

if ds_ssh_raw.longitude.max() > 180:
    ds_ssh_raw.coords['longitude'] = (ds_ssh_raw.coords['longitude'] + 180) % 360 - 180
    ds_ssh_raw = ds_ssh_raw.sortby(ds_ssh_raw.longitude)

ds_currents_range = pd.to_datetime(ds_currents.time.values)
ssh_raw_range = pd.to_datetime(ds_ssh_raw.time.values)

def map_ssh_to_all_years(date_range, ssh_date_range):
    mapped_times = [] 
    for target_date in date_range:
        target_day_of_year = target_date.dayofyear
        target_is_leap = pd.Timestamp(target_date.year, 12, 31).dayofyear == 366
        best_match = None
        min_diff = float('inf')        
        for ssh_date in ssh_date_range:
            ssh_day_of_year = ssh_date.dayofyear
            diff = abs(target_day_of_year - ssh_day_of_year)
            
            if diff < min_diff:
                min_diff = diff
                best_match = ssh_date
        
        mapped_times.append(best_match)
    
    return np.array(mapped_times, dtype='datetime64[ns]')

ssh_mapped_times = map_ssh_to_all_years(ds_currents_range, ssh_raw_range)
ds_currents_surface = ds_currents.isel(depth=0)

print(f"Dane prądów (powierzchnia): {len(ds_currents_range)} czasów")
print(f"Dane SSH (oryginalne): {len(ssh_raw_range)} czasów")
print("Mapowanie SSH na wszystkie lata zakończone.")

Wczytywanie danych prądów morskich (currents) i poziomu morza (sea_level)...
Dane prądów (powierzchnia): 875 czasów
Dane SSH (oryginalne): 365 czasów
Mapowanie SSH na wszystkie lata zakończone.


In [ ]:
print("Interpolacja prądów morskich")

t_currents = xr.DataArray(df_clean['Time'], dims="z")
lat_currents = xr.DataArray(df_clean['Lat_dd'], dims="z")
lon_currents = xr.DataArray(df_clean['Lon_dd'], dims="z")

ds_currents_interp = ds_currents_surface[['uo', 'vo']].interp(
    time=t_currents, 
    latitude=lat_currents, 
    longitude=lon_currents, 
    method='linear'
)
ds_currents_interp = ds_currents_interp.compute()

df_clean['Current_U_m_s'] = ds_currents_interp['uo'].values
df_clean['Current_V_m_s'] = ds_currents_interp['vo'].values
df_clean['Current_Speed_m_s'] = np.sqrt(df_clean['Current_U_m_s']**2 + df_clean['Current_V_m_s']**2)

print("Interpolacja poziomu morza (SSH)...")

ssh_extended_list = []
for orig_time, mapped_time in zip(pd.to_datetime(ds_currents.time.values), ssh_mapped_times):
    ssh_val = ds_ssh_raw.sel(time=mapped_time, method='nearest')
    ssh_extended_list.append(ssh_val)

ds_ssh = xr.concat(ssh_extended_list, dim='time')
ds_ssh['time'] = ('time', pd.to_datetime(ds_currents.time.values))

ds_ssh_interp = ds_ssh[['zos']].interp(
    time=t_currents,
    latitude=lat_currents,
    longitude=lon_currents,
    method='nearest'
)
ds_ssh_interp = ds_ssh_interp.compute()

df_clean['Sea_Level_SSH_m'] = ds_ssh_interp['zos'].values

kolumny_do_fillna = ['Current_U_m_s', 'Current_V_m_s', 'Current_Speed_m_s', 'Sea_Level_SSH_m']
for col in kolumny_do_fillna:
    df_clean[col] = df_clean[col].ffill().bfill()

print("\n--- Dane Prądów i SSH ---")
display(df_clean[['Time', 'Current_U_m_s', 'Current_V_m_s', 'Current_Speed_m_s', 'Sea_Level_SSH_m']].head())

print("\nPodsumowanie zbioru (Brakujące wartości):")
stats = df_clean[['Current_U_m_s', 'Current_V_m_s', 'Current_Speed_m_s', 'Sea_Level_SSH_m']].isna().sum()
print(stats)



Interpolacja prądów morskich...
Interpolacja poziomu morza (SSH)...

--- Dane Prądów i SSH (Pierwsze 5 wierszy) ---


,Time,Current_U_m_s,Current_V_m_s,Current_Speed_m_s,Sea_Level_SSH_m
0,2023-02-02 00:06:00,0.00787,0.030347,0.031351,-0.288
1,2023-02-02 00:21:00,0.00787,0.030347,0.031351,-0.288
2,2023-02-02 00:33:00,0.00787,0.030347,0.031351,-0.288
3,2023-02-02 00:54:00,0.00787,0.030347,0.031351,-0.288
4,2023-02-02 01:15:00,0.00787,0.030347,0.031351,-0.288



Podsumowanie zbioru (Brakujące wartości):
Current_U_m_s        0
Current_V_m_s        0
Current_Speed_m_s    0
Sea_Level_SSH_m      0
dtype: int64

✅ Sukces: Wszystkie dane prądów i SSH zostały zintegrowane.


In [ ]:

print("Obliczanie parametrów wiatru pozornego")
df_clean = calculate_awa_improved(df_clean)


Obliczanie parametrów wiatru pozornego...


In [ ]:

print("\n--- Sprawdzenie końcowe ---")
stats = df_clean[['Wave_Height_Sig', 'Wave_Dir_Mean', 'Wind_Speed_m_s', 'Current_Speed_m_s', 'Sea_Level_SSH_m', 'AWA_relative']].isna().sum()
print(stats.to_string())




--- Sprawdzenie końcowe (Brakujące wartości) ---
Wave_Height_Sig      0
Wave_Dir_Mean        0
Wind_Speed_m_s       0
Current_Speed_m_s    0
Sea_Level_SSH_m      0
AWA_relative         0

✅ Sukces: Brak pustych wartości. Dane są idealnie przygotowane do modelowania.


In [ ]:
import io
from scipy.interpolate import LinearNDInterpolator
print("Interpolacja mocy rotorów z wykresu biegunowego")

rotor_data_csv = """Angle,Wind_Speed,Power_kW
0,1,-2
45,1,4
90,1,14
135,1,10
180,1,-3
0,2,-8
45,2,16
90,2,56
135,2,40
180,2,-12
0,3,-18
45,3,36
90,3,126
135,3,90
180,3,-27
0,4,-32
45,4,64
90,4,224
135,4,160
180,4,-48
0,5,-50
45,5,100
90,5,350
135,5,250
180,5,-75
0,6,-68
45,6,140
90,6,450
135,6,340
180,6,-100
0,7,-85
45,7,185
90,7,560
135,7,440
180,7,-130
0,8,-105
45,8,235
90,8,660
135,8,530
180,8,-160
0,9,-125
45,9,290
90,9,750
135,9,610
180,9,-190
0,10,-150
45,10,350
90,10,850
135,10,700
180,10,-220
0,11,-175
45,11,410
90,11,960
135,11,780
180,11,-250
0,12,-200
45,12,470
90,12,1080
135,12,870
180,12,-285
0,13,-225
45,13,530
90,13,1200
135,13,950
180,13,-320
0,14,-250
45,14,590
90,14,1330
135,14,1030
180,14,-350
0,15,-280
45,15,650
90,15,1450
135,15,1100
180,15,-380
0,16,-310
45,16,710
90,16,1550
135,16,1180
180,16,-410
0,17,-340
45,17,770
90,17,1650
135,17,1260
180,17,-440
0,18,-370
45,18,830
90,18,1750
135,18,1340
180,18,-470
0,19,-400
45,19,890
90,19,1850
135,19,1420
180,19,-500
0,20,-430
45,20,950
90,20,1950
135,20,1500
180,20,-530
0,21,-460
45,21,1010
90,21,2050
135,21,1570
180,21,-560
0,22,-490
45,22,1070
90,22,2150
135,22,1640
180,22,-590
0,23,-520
45,23,1130
90,23,2220
135,23,1710
180,23,-620
0,24,-550
45,24,1190
90,24,2280
135,24,1780
180,24,-650
0,25,-580
45,25,1250
90,25,2350
135,25,1850
180,25,-680"""

df_rotor_map = pd.read_csv(io.StringIO(rotor_data_csv))
points = df_rotor_map[['Angle', 'Wind_Speed']].values
values = df_rotor_map['Power_kW'].values
rotor_interpolator = LinearNDInterpolator(points, values)

def calculate_total_rotor_power(row):
    wind_speed = row['Apparent_Wind_Speed_m_s']
    
    if wind_speed > 40:
        return 0.0
    wind_angle = abs(row['AWA_relative'])  
    wind_angle = wind_angle % 360
    if wind_angle > 180:
        wind_angle = 360 - wind_angle    
    wind_speed = np.clip(wind_speed, 1, 25)  
    power_1_rotor = rotor_interpolator(wind_angle, wind_speed)
    
    if np.isnan(power_1_rotor):
        power_1_rotor = 0.0
    total_power = power_1_rotor * 2
    
    if total_power < 0:
        total_power = 0
        
    return total_power

df_clean['Total_Rotor_Power_kW'] = df_clean.apply(calculate_total_rotor_power, axis=1)

print("\n--- Sprawdzenie końcowe ---")
stats = df_clean[['Wave_Height_Sig', 'Wave_Dir_Mean', 'Wind_Speed_m_s', 'Current_Speed_m_s', 'Sea_Level_SSH_m', 'AWA_relative']].isna().sum()
print(stats.to_string())

print(f"\n--- Warunki pracy rotorów ---")
print(f"Przypadki z wiatrem pozornym > 40 m/s (ROTORY WYŁĄCZONE): {(df_clean['Apparent_Wind_Speed_m_s'] > 40).sum()}")
print(f"Max prędkość wiatru pozornego: {df_clean['Apparent_Wind_Speed_m_s'].max():.2f} m/s")
print(f"Min prędkość wiatru pozornego: {df_clean['Apparent_Wind_Speed_m_s'].min():.2f} m/s")


Interpolacja mocy rotorów z wykresu biegunowego...

--- Sprawdzenie końcowe (Brakujące wartości) ---
Wave_Height_Sig      0
Wave_Dir_Mean        0
Wind_Speed_m_s       0
Current_Speed_m_s    0
Sea_Level_SSH_m      0
AWA_relative         0

--- Warunki pracy rotorów ---
Przypadki z wiatrem pozornym > 40 m/s (ROTORY WYŁĄCZONE): 0
Max prędkość wiatru pozornego: 22.74 m/s
Min prędkość wiatru pozornego: 0.02 m/s

✅ Sukces: Brak pustych wartości. Dane są idealnie przygotowane do modelowania.


In [ ]:
output_path = 'dane_gotowe_rotor.csv'
kolumny_do_zapisu = [
    'Time', 'Lat_dd', 'Lon_dd', 'Speed [kn]', 'Course [°]', 
    'Wind_Speed_m_s', 'Wind_U_10m', 'Wind_V_10m', 
    'Wave_Height_Sig', 'Wave_Dir_Mean', 'Wave_Dir_Sin', 'Wave_Dir_Cos',
    'Current_U_m_s', 'Current_V_m_s', 'Current_Speed_m_s',
    'Sea_Level_SSH_m',
    'AWA_relative', 'Apparent_Wind_Speed_m_s', 'Total_Rotor_Power_kW'
]

print(f"\nSummary of data with wind speed limits:")
print(f"Apparent wind speed > 40 m/s: {(df_clean['Apparent_Wind_Speed_m_s'] > 40).sum()} cases (rotors OFF)")

print(f"\nZapisano gotowy zbiór ({len(df_clean)} wierszy) do pliku: {output_path}")
df_clean[kolumny_do_zapisu].to_csv(output_path, index=False)


Summary of data with wind speed limits:


NameError: name 'df_clean' is not defined

In [ ]:
gotowe = pd.read_csv('dane_gotowe_rotor.csv')
gotowe.head()

,Time,Lat_dd,Lon_dd,Speed [kn],Course [°],Wind_Speed_m_s,Wind_U_10m,Wind_V_10m,Wave_Height_Sig,Wave_Dir_Mean,Wave_Dir_Sin,Wave_Dir_Cos,Current_U_m_s,Current_V_m_s,Current_Speed_m_s,Sea_Level_SSH_m,AWA_relative,Apparent_Wind_Speed_m_s,Total_Rotor_Power_kW
0,2023-02-02 00:06:00,56.82667,-0.12833,15.0,357.0,7.149395,7.112340,0.726958,2.089863,334.95886,-0.423269,0.906004,0.00787,0.030347,0.031351,-0.288,135.878082,10.256776,1400.887463
1,2023-02-02 00:21:00,56.88667,-0.12500,15.0,47.0,7.270172,7.243353,0.623888,2.152363,334.07605,-0.437178,0.899375,0.00787,0.030347,0.031351,-0.288,113.972876,4.906957,570.006967
2,2023-02-02 00:33:00,56.92333,-0.05833,16.0,46.0,7.425966,7.416633,0.372183,2.307535,330.84406,-0.487188,0.873297,0.00787,0.030347,0.031351,-0.288,118.368708,5.550920,680.474158
3,2023-02-02 00:54:00,56.98667,0.05500,15.0,44.0,7.656961,7.656569,-0.077478,2.307535,330.84406,-0.487188,0.873297,0.00787,0.030347,0.031351,-0.288,113.806743,6.078720,800.929924
4,2023-02-02 01:15:00,57.05000,0.17500,16.0,46.0,7.617507,7.587544,-0.674967,2.307535,330.84406,-0.487188,0.873297,0.00787,0.030347,0.031351,-0.288,119.388324,6.606439,884.550049


In [ ]:
okno_pogodowe = 10
MAIN_ENGINE_KW = 5600

teoretyczna_SOG_bez_limitu = np.where(
    gotowe['Speed [kn]'] > 0,
    gotowe['Speed [kn]'] * np.cbrt((MAIN_ENGINE_KW + gotowe['Total_Rotor_Power_kW']) / MAIN_ENGINE_KW),
    0
)

surowy_zysk_w_wezlach = teoretyczna_SOG_bez_limitu - gotowe['Speed [kn]']

zysk_z_limitami = np.clip(surowy_zysk_w_wezlach, 0, 6.0)

gotowe['Delta_SOG_kn'] = zysk_z_limitami
gotowe['Theoretical_SOG_Rotors_kn'] = gotowe['Speed [kn]'] + zysk_z_limitami

print("\nPodgląd przyrostu prędkości statku:")
display(gotowe[['Speed [kn]', 'Total_Rotor_Power_kW', 'Delta_SOG_kn', 'Theoretical_SOG_Rotors_kn']].head())


Podgląd przyrostu prędkości statku:


,Speed [kn],Total_Rotor_Power_kW,Delta_SOG_kn,Theoretical_SOG_Rotors_kn
0,15.0,1400.887463,1.158943,16.158943
1,15.0,570.006967,0.492582,15.492582
2,16.0,680.474158,0.623461,16.623461
3,15.0,800.929924,0.683498,15.683498
4,16.0,884.550049,0.801598,16.801598


In [ ]:


gotowe['Window_Base_10kn'] = gotowe['Speed [kn]'] >= okno_pogodowe 
gotowe['Window_Rotor_10kn'] = gotowe['Theoretical_SOG_Rotors_kn'] >= okno_pogodowe 
gotowe['Window_Base_10kn'] = gotowe['Window_Base_10kn'].astype(int)
gotowe['Window_Rotor_10kn'] = gotowe['Window_Rotor_10kn'].astype(int)


punkty_baza = gotowe['Window_Base_10kn'].sum()
punkty_rotory = gotowe['Window_Rotor_10kn'].sum()
zysk_punktow = punkty_rotory - punkty_baza

print(f"Liczba pomiarów spełniających warunek okna (SOG >= 10 kn):")
print(f" -> Scenariusz bez rotorów: {punkty_baza}")
print(f" -> Scenariusz z rotorami:  {punkty_rotory}")

if zysk_punktow > 0:
    procent = (zysk_punktow / punkty_baza) * 100 if punkty_baza > 0 else 0
    print(f"WNIOSEK: Dzięki rotorom statek zyskał {zysk_punktow} dodatkowych punktów pomiarowych w oknie pogodowym (+{procent:.2f}% czasu operacyjnego).")
else:
    print("ℹBrak zwiększenia okien pogodowych przy progu 10 węzłów dla tych danych.")
    
display(gotowe[['Theoretical_SOG_Rotors_kn', 'Window_Base_10kn', 'Window_Rotor_10kn']].head(10))

Liczba pomiarów spełniających warunek okna (SOG >= 10 kn):
 -> Scenariusz bez rotorów: 97976
 -> Scenariusz z rotorami:  102310
✅ WNIOSEK: Dzięki rotorom statek zyskał 4334 dodatkowych punktów pomiarowych w oknie pogodowym (+4.42% czasu operacyjnego!).


,Theoretical_SOG_Rotors_kn,Window_Base_10kn,Window_Rotor_10kn
0,16.158943,1,1
1,15.492582,1,1
2,16.623461,1,1
3,15.683498,1,1
4,16.801598,1,1
5,16.827895,1,1
6,16.865992,1,1
7,17.131705,1,1
8,16.768670,1,1
9,16.880636,1,1


In [ ]:
print("\nAnaliza Okien Pogodowych")

PRÓG_PRĘDKOŚCI = 10.0
gotowe['Time'] = pd.to_datetime(gotowe['Time'])
gotowe = gotowe.sort_values('Time')
gotowe['Time_Delta_Hours'] = gotowe['Time'].diff().dt.total_seconds() / 3600.0
gotowe['Time_Delta_Hours'] = gotowe['Time_Delta_Hours'].fillna(0)
gotowe['Time_Delta_Hours'] = np.where(gotowe['Time_Delta_Hours'] > 1, 0, gotowe['Time_Delta_Hours'])
gotowe['Window_Base_10kn'] = (gotowe['Speed [kn]'] >= PRÓG_PRĘDKOŚCI).astype(int)
gotowe['Window_Rotor_10kn'] = (gotowe['Theoretical_SOG_Rotors_kn'] >= PRÓG_PRĘDKOŚCI).astype(int)
godziny_baza = (gotowe['Window_Base_10kn'] * gotowe['Time_Delta_Hours']).sum()
godziny_rotory = (gotowe['Window_Rotor_10kn'] * gotowe['Time_Delta_Hours']).sum()
zysk_godzin = godziny_rotory - godziny_baza

print(f"Rzeczywisty czas operacyjny (SOG >= 10 kn):")
print(f" -> Scenariusz bez rotorów: {godziny_baza:.1f} godzin")
print(f" -> Scenariusz z rotorami:  {godziny_rotory:.1f} godzin")

if zysk_godzin > 0:
    procent = (zysk_godzin / godziny_baza) * 100 if godziny_baza > 0 else 0
    print(f"WNIOSEK BIZNESOWY: Dzięki rotorom statek zyskał {zysk_godzin:.1f} dodatkowych godzin operacyjnych (+{procent:.2f}% czasu w oknie pogodowym).")
else:
    print("ℹBrak zwiększenia okien pogodowych przy progu 10 węzłów dla tych danych.")


Analiza Okien Pogodowych (w oparciu o czas operacyjny)...
Rzeczywisty czas operacyjny (SOG >= 10 kn):
 -> Scenariusz bez rotorów: 13513.3 godzin
 -> Scenariusz z rotorami:  14105.2 godzin
✅ WNIOSEK BIZNESOWY: Dzięki rotorom statek zyskał 591.9 dodatkowych godzin operacyjnych (+4.38% czasu w oknie pogodowym).


In [ ]:
print("\nObliczanie zaoszczędzonego czasu podróży...")

gotowe['Distance_NM'] = gotowe['Speed [kn]'] * gotowe['Time_Delta_Hours']

gotowe['Time_Rotor_Hours'] = np.where(
    gotowe['Theoretical_SOG_Rotors_kn'] > 0,
    gotowe['Distance_NM'] / gotowe['Theoretical_SOG_Rotors_kn'],
    gotowe['Time_Delta_Hours'] 
)

gotowe['Time_Saved_Hours'] = gotowe['Time_Delta_Hours'] - gotowe['Time_Rotor_Hours']

calkowity_czas_bazowy = gotowe['Time_Delta_Hours'].sum()
calkowity_czas_rotory = gotowe['Time_Rotor_Hours'].sum()
calkowity_zysk_czasu = gotowe['Time_Saved_Hours'].sum()
calkowity_dystans = gotowe['Distance_NM'].sum()

print("-" * 50)
print("🚢 PODSUMOWANIE WPŁYWU ROTORÓW NA TRASIE")
print("-" * 50)
print(f"Całkowity pokonany dystans:        {calkowity_dystans:.1f} NM (Mil Morskich)")
print(f"Całkowity czas żeglugi (silnik):   {calkowity_czas_bazowy:.1f} godzin")
print(f"Całkowity czas żeglugi (rotory):   {calkowity_czas_rotory:.1f} godzin")
print(f"💰 Zaoszczędzony czas:              {calkowity_zysk_czasu:.1f} godzin!")

if calkowity_czas_bazowy > 0:
    procent_czasu = (calkowity_zysk_czasu / calkowity_czas_bazowy) * 100
    print(f"\n✅ WNIOSEK: Dzięki zastosowaniu rotorów Flettnera, czas trwania wszystkich podróży skrócił się o {procent_czasu:.2f}%.")


Obliczanie zaoszczędzonego czasu podróży...
--------------------------------------------------
🚢 PODSUMOWANIE WPŁYWU ROTORÓW NA TRASIE
--------------------------------------------------
Całkowity pokonany dystans:        200674.5 NM (Mil Morskich)
Całkowity czas żeglugi (silnik):   18320.4 godzin
Całkowity czas żeglugi (rotory):   17720.0 godzin
💰 Zaoszczędzony czas:              600.4 godzin!

✅ WNIOSEK: Dzięki zastosowaniu rotorów Flettnera, czas trwania wszystkich podróży skrócił się o 3.28%.


In [ ]:
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
import xgboost as xgb

target = 'Speed [kn]'
wszystkie_kolumny = gotowe.columns.tolist()
cechy_podstawowe = ['Wind_Speed_m_s', 'Wave_Height_Sig', 'Wave_Dir_Sin', 'Wave_Dir_Cos', 
                    'Apparent_Wind_Speed_m_s', 'AWA_relative', 'Total_Rotor_Power_kW','Current_U_m_s', 'Current_V_m_s', 'Sea_Level_SSH_m']

gotowe_ml = gotowe.dropna(subset=cechy_podstawowe + [target]).copy()

gotowe_ml = gotowe_ml.sort_values('Time')
split_index = int(len(gotowe_ml) * 0.8)


train_data = gotowe_ml.iloc[:split_index]
test_data = gotowe_ml.iloc[split_index:]

y_train = train_data[target]


print(f"Zbiór Treningowy (Do pracy): {len(train_data)} wierszy")
print(f"Zbiór Testowy (ZAMKNIĘTY W SZAFIE): {len(test_data)} wierszy\n")


tscv = TimeSeriesSplit(n_splits=3)


def evaluate_baseline_cv(model, X, y, cv_splitter):
    scores = cross_validate(model, X, y, cv=cv_splitter, 
                            scoring=('neg_mean_absolute_error', 'neg_root_mean_squared_error', 'r2'),
                            n_jobs=-1)
    return {
        "MAE (węzły)": round(-scores['test_neg_mean_absolute_error'].mean(), 3),
        "RMSE": round(-scores['test_neg_root_mean_squared_error'].mean(), 3),
        "R2": round(scores['test_r2'].mean(), 3)
    }

modele_konfiguracja = {
    "Baseline 1 (Średnia)": {
        "algorytm": DummyRegressor(strategy="mean"),
        "params": None,
        "cechy": cechy_podstawowe
    },
    "Baseline 2 (Regresja)": {
        "algorytm": LinearRegression(),
        "params": None,
        "cechy": ['Wind_Speed_m_s', 'Wave_Height_Sig']
    },
    "Random Forest (Randomized)": {
        "algorytm": RandomForestRegressor(random_state=42, n_jobs=-1),
        "params": {
            'n_estimators': [200, 500],             
            'max_depth': [10, 15, 20, 25, None],         
            'min_samples_split': [2, 5, 10],            
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', 1.0]       
        },
        "cechy": cechy_podstawowe
    },
    "XGBoost (Randomized)": {
        "algorytm": xgb.XGBRegressor(random_state=42, n_jobs=-1, objective='reg:squarederror'),
        "params": {
            'n_estimators': [500, 1000],            
            'learning_rate': [0.01, 0.05, 0.1, 0.2],    
            'max_depth': [5, 7, 10, 12, 15],             
            'subsample': [0.6, 0.8, 1.0],               
            'colsample_bytree': [0.6, 0.8, 1.0],         
            'gamma': [0, 0.1, 0.5, 1.0]                  
        },
        "cechy": cechy_podstawowe
    }
}

metryki_grid = {
    'MAE': 'neg_mean_absolute_error',
    'RMSE': 'neg_root_mean_squared_error',
    'R2': 'r2'
}

wyniki_cv = []
wytrenowane_modele = {}

for nazwa, konfig in modele_konfiguracja.items():
    print(f"Trenowanie")
    baza_model = konfig["algorytm"]
    cechy_modelu = konfig["cechy"]
    
    X_train_model = train_data[cechy_modelu]
    
    if konfig["params"] is not None:
        random_search = RandomizedSearchCV(
            estimator=baza_model,
            param_distributions=konfig["params"],
            n_iter=15,                    
            cv=tscv,
            scoring=metryki_grid,
            refit='MAE',                   
            n_jobs=-1,
            random_state=42,               
            verbose=1                    
        )
        random_search.fit(X_train_model, y_train)
        
        best_index = random_search.best_index_
        wyniki_cv.append({
            "Model": nazwa,
            "MAE (węzły)": round(-random_search.cv_results_['mean_test_MAE'][best_index], 3),
            "RMSE": round(-random_search.cv_results_['mean_test_RMSE'][best_index], 3),
            "R2": round(random_search.cv_results_['mean_test_R2'][best_index], 3)
        })
        
        wytrenowane_modele[nazwa] = random_search.best_estimator_
        
    else:
        metryki_cv = evaluate_baseline_cv(baza_model, X_train_model, y_train, tscv)
        metryki_cv["Model"] = nazwa
        wyniki_cv.append(metryki_cv)
        wytrenowane_modele[nazwa] = baza_model.fit(X_train_model, y_train)

df_wyniki_cv = pd.DataFrame(wyniki_cv)[['Model', 'MAE (węzły)', 'RMSE', 'R2']]

display(df_wyniki_cv)

best = df_wyniki_cv.sort_values('MAE (węzły)').iloc[0]
v_avg = y_train.mean()

error_pct = (best['MAE (węzły)'] / v_avg) * 100
conf_95_pct = (2 * best['RMSE'] / v_avg) * 100

print(f"\n--- OCENA NIEPEWNOŚCI ({best['Model']}) ---")
print(f"Średni błąd modelu: {error_pct:.2f}%")
print(f"Przedział ufności (95%): ±{conf_95_pct:.2f}%")
print(f"Interpretacja: Model przewiduje SOG z dokładnością do ok. {best['MAE (węzły)']} kn.")

NameError: name 'gotowe' is not defined

In [ ]:
import joblib

print("="*50)
print("PAKOWANIE MODELU DO WDROŻENIA (DEPLOYMENT)")
print("="*50)

ostateczny_model = wytrenowane_modele["XGBoost (Randomized)"]

plik_modelu = 'xgboost_model_rotory.joblib'
plik_cech = 'xgboost_lista_cech.joblib'

joblib.dump(ostateczny_model, plik_modelu)
joblib.dump(cechy_podstawowe, plik_cech)


📦 PAKOWANIE MODELU DO WDROŻENIA (DEPLOYMENT)
✅ SUKCES! Zapisano model ML do pliku: xgboost_model_rotory.joblib
✅ SUKCES! Zapisano wymaganą listę cech do pliku: xgboost_lista_cech.joblib
Teraz możesz wysłać te dwa pliki komu tylko chcesz!


In [ ]:

def ocen_oplacalnosc_rotorow(sciezka_do_nowych_danych, dystans_trasy_nm=None):
    try:
        model = joblib.load('xgboost_model_rotory.joblib')
        wymagane_cechy = joblib.load('xgboost_lista_cech.joblib')
    except FileNotFoundError:
        return "Błąd: Brak plików modelu! Upewnij się, że masz pliki .joblib w folderze."
    
    print("Analiza warunków na trasie")
    dane = pd.read_csv(sciezka_do_nowych_danych)
    
    X_dane = dane[wymagane_cechy].copy()
    X_dane = X_dane.fillna(0)
    
    predkosc_z_rotorami = model.predict(X_dane)
    
    X_dane_bez_rotorow = X_dane.copy()
    X_dane_bez_rotorow['Total_Rotor_Power_kW'] = 0.0
    
    predkosc_bez_rotorow = model.predict(X_dane_bez_rotorow)
    
    zysk_wezly = predkosc_z_rotorami - predkosc_bez_rotorow
    zysk_wezly = np.clip(zysk_wezly, 0, None) 
    
    sredni_zysk = zysk_wezly.mean()
    
    print("\n" + "="*50)
    print("RAPORT OPŁACALNOŚCI")
    print("="*50)
    print(f"Średnia przewidywana prędkość BEZ rotorów: {predkosc_bez_rotorow.mean():.2f} węzłów")
    print(f"Średnia przewidywana prędkość Z ROTORAMI:  {predkosc_z_rotorami.mean():.2f} węzłów")
    print(f"📈 Średni udowodniony zysk na trasie:     +{sredni_zysk:.2f} węzłów")
    
    if dystans_trasy_nm:
        czas_bez = dystans_trasy_nm / predkosc_bez_rotorow.mean()
        czas_z = dystans_trasy_nm / predkosc_z_rotorami.mean()
        zysk_czasu = czas_bez - czas_z
        
        print("-" * 50)
        print(f"DLA TRASY O DŁUGOŚCI {dystans_trasy_nm} NM:")
        print(f"Czas rejsu bez rotorów: {czas_bez:.1f} godzin")
        print(f"Czas rejsu z rotorami:  {czas_z:.1f} godzin")
        print(f"ZAOSZCZĘDZONY CZAS:    {zysk_czasu:.1f} godzin!")
        
    
            
    dane['Przewidywane_SOG_Bez_Rotorow'] = predkosc_bez_rotorow
    dane['Przewidywane_SOG_Z_Rotorami'] = predkosc_z_rotorami
    dane['Czysty_Zysk_Wezly'] = zysk_wezly
    dane.to_csv('raport_inwestorski_detale.csv', index=False)
    
    return dane
